# LeetCode 4048 — Count Values With Equally Spaced Occurrences I bằng Python

Link đề: https://leetcode.com/problems/count-values-with-equally-spaced-occurrences-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Biweekly Contest 191, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập duyệt mọi bộ ba chỉ số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một số `x` là **đặc biệt (special)** nếu nó xuất hiện **đúng 3 lần** trong `nums` **và** ba vị trí `i1 < i2 < i3` của nó **cách đều nhau** (`i2 − i1 = i3 − i2`); đếm số giá trị đặc biệt **khác nhau**.
2. Biết cách **gom vị trí theo giá trị** bằng `dict` (hash table): `x -> [các chỉ số của x]`, và dùng `enumerate` để lấy cả chỉ số lẫn giá trị.
3. Đổi điều kiện "cách đều" thành biểu thức dễ kiểm tra: `i1 + i3 == 2 * i2`.
4. Đi từ **brute force quét lại mảng cho từng giá trị** → **dict các danh sách chỉ số** → **đếm một lượt, chỉ nhớ vài biến** → **sắp xếp + `groupby`**.
5. Tránh các lỗi kinh điển: dùng "ít nhất 3 lần" thay vì "đúng 3 lần", đọc `pos[2]` khi danh sách ngắn hơn 3, so sánh **giá trị** thay vì **chỉ số**, và đếm số lần xuất hiện thay vì số giá trị khác nhau.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Một số nguyên `x` được gọi là **đặc biệt** nếu:

- `x` xuất hiện **đúng ba lần** trong `nums`;
- cả ba lần xuất hiện **cách đều nhau**: nếu các chỉ số xuất hiện là `i1 < i2 < i3` thì `i2 − i1 = i3 − i2`.

Trả về **số lượng số nguyên đặc biệt khác nhau** trong `nums`.

**Ví dụ 1**

- Input: `nums = [1,8,1,5,1,5,8,5]`
- Output: `2`
- Giải thích: `1` đặc biệt vì xuất hiện đúng 3 lần ở chỉ số `0, 2, 4` (cách đều 2). `5` đặc biệt vì xuất hiện đúng 3 lần ở chỉ số `3, 5, 7`. `8` không đặc biệt vì chỉ xuất hiện 2 lần.

**Ví dụ 2**

- Input: `nums = [8,8,8,8]`
- Output: `0`
- Giải thích: `8` xuất hiện 4 lần, không phải đúng 3 lần.

**Ví dụ 3**

- Input: `nums = [8,6,6,8,8]`
- Output: `0`
- Giải thích: `8` xuất hiện ở chỉ số `0, 3, 4` — khoảng cách `3` và `1`, không đều. `6` chỉ xuất hiện 2 lần.

**Ràng buộc:** `3 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
import random
from collections import Counter, defaultdict
from itertools import groupby

examples = [
    ([1, 8, 1, 5, 1, 5, 8, 5], 2),
    ([8, 8, 8, 8], 0),
    ([8, 6, 6, 8, 8], 0),
]
for nums, expected in examples:
    print(f"nums={nums}, kỳ vọng {expected}")

nums=[1, 8, 1, 5, 1, 5, 8, 5], kỳ vọng 2
nums=[8, 8, 8, 8], kỳ vọng 0
nums=[8, 6, 6, 8, 8], kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Chỉ số và `enumerate`

Mỗi phần tử của list có **chỉ số** (index) bắt đầu từ `0`. Bài này quan tâm đến **vị trí** của giá trị, nên ta cần duyệt cả chỉ số lẫn giá trị. `enumerate(nums)` sinh ra các cặp `(chỉ số, giá trị)`, và ta "bung" cặp bằng `for i, x in enumerate(nums)`.

In [2]:
nums = [1, 8, 1, 5]
print(list(enumerate(nums)))            # [(0, 1), (1, 8), (2, 1), (3, 5)]
for i, x in enumerate(nums):
    print(f"chỉ số {i} chứa giá trị {x}")

[(0, 1), (1, 8), (2, 1), (3, 5)]
chỉ số 0 chứa giá trị 1
chỉ số 1 chứa giá trị 8
chỉ số 2 chứa giá trị 1
chỉ số 3 chứa giá trị 5


### 3.2. `dict` — bảng tra cứu "giá trị → danh sách chỉ số"

`dict` lưu cặp **key → value**. Ở đây `key` là giá trị trong `nums`, còn `value` là **list các chỉ số** nơi nó xuất hiện. Thêm phần tử vào list bằng `append`. Kiểm tra key đã có chưa bằng `x in d`.

In [3]:
pos = {}
for i, x in enumerate([1, 8, 1, 5, 1]):
    if x not in pos:
        pos[x] = []          # lần đầu thấy x: tạo list rỗng
    pos[x].append(i)         # thêm chỉ số i vào list của x
print(pos)                   # {1: [0, 2, 4], 8: [1], 5: [3]}
print(1 in pos, 7 in pos)    # True False
print(len(pos[1]))           # 3: số lần 1 xuất hiện

{1: [0, 2, 4], 8: [1], 5: [3]}
True False
3


### 3.3. `defaultdict(list)` — bỏ bước "tạo list rỗng"

Viết `if x not in pos: pos[x] = []` rất dài dòng. `collections.defaultdict(list)` tự tạo list rỗng khi ta truy cập key chưa có.

In [4]:
g = defaultdict(list)
for i, x in enumerate([1, 8, 1, 5, 1]):
    g[x].append(i)           # không cần kiểm tra x đã có chưa
print(dict(g))               # {1: [0, 2, 4], 8: [1], 5: [3]}
print(list(g.values()))      # [[0, 2, 4], [1], [3]]

{1: [0, 2, 4], 8: [1], 5: [3]}
[[0, 2, 4], [1], [3]]


### 3.4. "Cách đều" ⇔ `i1 + i3 == 2 * i2`

Ba chỉ số `i1 < i2 < i3` cách đều khi `i2 − i1 = i3 − i2`. Chuyển vế: `i1 + i3 = 2·i2`. Biểu thức sau chỉ cần một phép nhân và một phép cộng. Ví dụ `0, 2, 4`: `0 + 4 = 4 = 2·2` (đều); `0, 3, 4`: `0 + 4 = 4` nhưng `2·3 = 6` (không đều).

In [5]:
for triple in [(0, 2, 4), (3, 5, 7), (0, 3, 4), (1, 2, 5)]:
    i1, i2, i3 = triple
    print(triple, "khoảng cách:", i2 - i1, i3 - i2, "| cách đều?", i1 + i3 == 2 * i2)

(0, 2, 4) khoảng cách: 2 2 | cách đều? True
(3, 5, 7) khoảng cách: 2 2 | cách đều? True
(0, 3, 4) khoảng cách: 3 1 | cách đều? False
(1, 2, 5) khoảng cách: 1 3 | cách đều? False


### 3.5. `Counter` đếm số lần xuất hiện

`Counter(nums)` trả về dict-like `giá trị -> số lần`. Dùng khi chỉ cần biết "bao nhiêu lần", không cần vị trí.

In [6]:
c = Counter([1, 8, 1, 5, 1, 5, 8, 5])
print(c)                                # Counter({1: 3, 5: 3, 8: 2})
print([x for x, cnt in c.items() if cnt == 3])   # các giá trị xuất hiện đúng 3 lần: [1, 5]

Counter({1: 3, 5: 3, 8: 2})
[1, 5]


### 3.6. `sorted` theo cặp và `groupby`

`sorted` với list các cặp `(giá trị, chỉ số)` sắp xếp theo giá trị trước, chỉ số sau. `itertools.groupby(iterable, key)` gom các phần tử **liên tiếp** có cùng key; vì vậy phải **sắp xếp trước**.

In [7]:
pairs = sorted((x, i) for i, x in enumerate([1, 8, 1, 5, 1, 5, 8, 5]))
print(pairs)    # [(1, 0), (1, 2), (1, 4), (5, 3), (5, 5), (5, 7), (8, 1), (8, 6)]
for x, grp in groupby(pairs, key=lambda p: p[0]):
    print(x, [i for _, i in grp])

[(1, 0), (1, 2), (1, 4), (5, 3), (5, 5), (5, 7), (8, 1), (8, 6)]
1 [0, 2, 4]
5 [3, 5, 7]
8 [1, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1: `nums = [1,8,1,5,1,5,8,5]`. Viết chỉ số phía trên:

```
chỉ số: 0 1 2 3 4 5 6 7
nums  : 1 8 1 5 1 5 8 5
```

Với mỗi giá trị, ghi các chỉ số xuất hiện:

| giá trị | các chỉ số | số lần | `i1 + i3` vs `2·i2` | đặc biệt? |
|---|---|---|---|---|
| 1 | 0, 2, 4 | 3 | `0 + 4 = 4`, `2·2 = 4` | có |
| 5 | 3, 5, 7 | 3 | `3 + 7 = 10`, `2·5 = 10` | có |
| 8 | 1, 6 | 2 | (không đủ 3 lần) | không |

Vậy đáp án `2`. Quy trình: **gom vị trí theo giá trị → kiểm tra "đúng 3 lần" → kiểm tra cách đều**.

In [8]:
nums = [1, 8, 1, 5, 1, 5, 8, 5]
for x in sorted(set(nums)):
    positions = [i for i, v in enumerate(nums) if v == x]
    status = "đặc biệt" if len(positions) == 3 and positions[0] + positions[2] == 2 * positions[1] else "không"
    print(f"x={x}: chỉ số {positions} -> {status}")

x=1: chỉ số [0, 2, 4] -> đặc biệt
x=5: chỉ số [3, 5, 7] -> đặc biệt
x=8: chỉ số [1, 6] -> không


## 5. Cách 1 — Brute Force / baseline: quét lại mảng cho từng giá trị

**Ý tưởng.** Lấy tập các giá trị khác nhau `set(nums)`. Với **mỗi** giá trị `x`, quét lại cả mảng để lấy list chỉ số `positions` của nó, rồi kiểm tra: độ dài đúng 3 và `positions[0] + positions[2] == 2 * positions[1]`.

Ta quét lại mảng `d` lần với `d` là số giá trị khác nhau — chưa tận dụng được việc "gom một lượt".

In [9]:
def count_special_bruteforce(nums: list[int]) -> int:
    answer = 0
    for x in set(nums):                                   # mỗi giá trị khác nhau
        positions = []
        for i in range(len(nums)):                        # quét lại toàn mảng
            if nums[i] == x:
                positions.append(i)
        if len(positions) == 3 and positions[0] + positions[2] == 2 * positions[1]:
            answer += 1
    return answer


for nums, expected in examples:
    print(f"nums={nums} -> {count_special_bruteforce(nums)} (kỳ vọng {expected})")

nums=[1, 8, 1, 5, 1, 5, 8, 5] -> 2 (kỳ vọng 2)
nums=[8, 8, 8, 8] -> 0 (kỳ vọng 0)
nums=[8, 6, 6, 8, 8] -> 0 (kỳ vọng 0)


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)` và `d` là số giá trị khác nhau (`d ≤ n`). Mỗi giá trị quét `n` phần tử: **Time `O(n·d)`**, tệ nhất `O(n²)` khi các giá trị đều khác nhau. **Space `O(d)`** cho tập `set(nums)` (cộng một list tạm tối đa 3 phần tử trong trường hợp cần).

**Nút thắt.** Việc quét lại mảng cho mỗi giá trị là thừa: ta có thể đi qua mảng **một lần** và ghi nhớ vị trí của mọi giá trị cùng lúc.

## 6. Ý tưởng thuật toán / bước đột phá

**Đi qua mảng một lần, ghi nhớ vị trí theo giá trị.** Khi gặp `nums[i] = x`, thêm `i` vào danh sách của `x`. Cuối cùng mỗi giá trị đã có sẵn danh sách chỉ số **đã sắp tăng** (vì ta duyệt `i` từ trái sang phải). Lúc đó:

- đúng 3 lần ⇔ `len(pos) == 3`;
- cách đều ⇔ `pos[0] + pos[2] == 2 * pos[1]`.

Thậm chí không cần lưu cả danh sách: chỉ cần nhớ **lần xuất hiện đầu, khoảng cách giữa lần 1 và lần 2, và số lần đã thấy** (Cách 3). Khi số lần đã thấy vượt 3 thì giá trị đó bị loại vĩnh viễn.

## 7. Các cách tiếp theo

### Cách 2 — `dict` các danh sách chỉ số (hash table)

**Giải thích.**

1. Tạo `g = defaultdict(list)`.
2. Duyệt `for i, x in enumerate(nums)` và `g[x].append(i)`.
3. Với mỗi `pos` trong `g.values()`, đếm 1 nếu `len(pos) == 3` và `pos[0] + pos[2] == 2 * pos[1]`.

Chú ý thứ tự kiểm tra: **`len(pos) == 3` đứng trước** nhờ `and` ngắn mạch, nên không bao giờ đọc `pos[2]` của list ngắn.

In [10]:
def count_special_dict(nums: list[int]) -> int:
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)                      # gom chỉ số theo giá trị trong một lượt
    answer = 0
    for pos in g.values():
        if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1]:
            answer += 1
    return answer


for nums, expected in examples:
    print(f"nums={nums} -> {count_special_dict(nums)} (kỳ vọng {expected})")

nums=[1, 8, 1, 5, 1, 5, 8, 5] -> 2 (kỳ vọng 2)
nums=[8, 8, 8, 8] -> 0 (kỳ vọng 0)
nums=[8, 6, 6, 8, 8] -> 0 (kỳ vọng 0)


In [11]:
# Code nộp LeetCode cho Cách 2
from collections import defaultdict
from typing import List


class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        g = defaultdict(list)
        for i, x in enumerate(nums):
            g[x].append(i)
        return sum(
            len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1] for pos in g.values()
        )

**Độ phức tạp Cách 2:** **Time `O(n)`** (một lượt `append`, một lượt duyệt `values`), **Space `O(n)`** cho các danh sách (tổng độ dài các danh sách đúng bằng `n`). Thao tác `dict` là `O(1)` trung bình.

### Cách 3 — Đếm một lượt, chỉ nhớ vài con số

**Giải thích.** Với mỗi giá trị `x` ta lưu `count[x]`, `first[x]` (chỉ số lần 1), `gap[x]` (khoảng cách lần 2 − lần 1), và cờ `ok[x]`:

- lần xuất hiện thứ 1 tại `i`: `first[x] = i`;
- lần thứ 2: `gap[x] = i − first[x]`, `last[x] = i`;
- lần thứ 3: `ok[x] = (i − last[x] == gap[x])`;
- lần thứ 4 trở lên: `count` vượt 3, giá trị này bị loại (vì `count[x] != 3` cuối cùng).

Kết quả: số `x` có `count[x] == 3` và `ok[x]`. Không lưu danh sách, chỉ vài biến mỗi giá trị.

In [12]:
def count_special_counters(nums: list[int]) -> int:
    count, first, last, gap, ok = {}, {}, {}, {}, {}
    for i, x in enumerate(nums):
        count[x] = count.get(x, 0) + 1      # .get(x, 0): nếu x chưa có thì coi như 0
        c = count[x]
        if c == 1:
            first[x] = i
            last[x] = i
        elif c == 2:
            gap[x] = i - first[x]
            last[x] = i
        elif c == 3:
            ok[x] = (i - last[x] == gap[x])
            last[x] = i
        # c >= 4: không cần làm gì, count[x] != 3 nên x sẽ bị loại
    return sum(1 for x in count if count[x] == 3 and ok.get(x, False))


for nums, expected in examples:
    print(f"nums={nums} -> {count_special_counters(nums)} (kỳ vọng {expected})")

nums=[1, 8, 1, 5, 1, 5, 8, 5] -> 2 (kỳ vọng 2)
nums=[8, 8, 8, 8] -> 0 (kỳ vọng 0)
nums=[8, 6, 6, 8, 8] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(d)`** với `d ≤ min(n, 100)` là số giá trị khác nhau (các dict có không quá `d` khóa; không còn danh sách chỉ số dài).

### Cách 4 — Sắp xếp rồi `groupby`

**Giải thích.** Tạo list cặp `(x, i)` rồi `sorted`: các cặp cùng giá trị nằm cạnh nhau, và trong nhóm các chỉ số đã tăng dần. Dùng `groupby` tách nhóm; với mỗi nhóm lấy list chỉ số và kiểm tra như Cách 2. Không dùng `dict`, thay bằng sắp xếp.

In [13]:
def count_special_sort(nums: list[int]) -> int:
    pairs = sorted((x, i) for i, x in enumerate(nums))
    answer = 0
    for _, group in groupby(pairs, key=lambda p: p[0]):
        pos = [i for _, i in group]
        if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1]:
            answer += 1
    return answer


for nums, expected in examples:
    print(f"nums={nums} -> {count_special_sort(nums)} (kỳ vọng {expected})")

nums=[1, 8, 1, 5, 1, 5, 8, 5] -> 2 (kỳ vọng 2)
nums=[8, 8, 8, 8] -> 0 (kỳ vọng 0)
nums=[8, 6, 6, 8, 8] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (do sắp xếp), **Space `O(n)`**. Chậm hơn dict một chút nhưng không cần băm; có ích khi dữ liệu đã được sắp theo giá trị.

> Cách 2 là cách "chuẩn" vì rõ ràng nhất và `O(n)`. Cách 3 tiết kiệm bộ nhớ khi mỗi giá trị xuất hiện rất nhiều lần.

## 8. TRACE từng bước

Ta theo dõi Cách 2 trên `nums = [8,6,6,8,8]` (ví dụ 3, có giá trị đúng 3 lần nhưng không cách đều), rồi ví dụ 1.

In [14]:
def trace_dict(nums):
    print("nums =", nums)
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
        print(f"  i={i}, x={x} -> g = {dict(g)}")
    answer = 0
    for x, pos in g.items():
        if len(pos) != 3:
            print(f"  x={x}: pos={pos} có {len(pos)} lần != 3 -> không đặc biệt")
            continue
        left, right = pos[0] + pos[2], 2 * pos[1]
        even = left == right
        print(f"  x={x}: pos={pos}, i1+i3={left}, 2*i2={right} -> {'cách đều, ĐẶC BIỆT' if even else 'không đều'}")
        answer += even
    print("  => đáp án:", answer)
    return answer


assert trace_dict([8, 6, 6, 8, 8]) == 0
print()
assert trace_dict([1, 8, 1, 5, 1, 5, 8, 5]) == 2

nums = [8, 6, 6, 8, 8]
  i=0, x=8 -> g = {8: [0]}
  i=1, x=6 -> g = {8: [0], 6: [1]}
  i=2, x=6 -> g = {8: [0], 6: [1, 2]}
  i=3, x=8 -> g = {8: [0, 3], 6: [1, 2]}
  i=4, x=8 -> g = {8: [0, 3, 4], 6: [1, 2]}
  x=8: pos=[0, 3, 4], i1+i3=4, 2*i2=6 -> không đều
  x=6: pos=[1, 2] có 2 lần != 3 -> không đặc biệt
  => đáp án: 0

nums = [1, 8, 1, 5, 1, 5, 8, 5]
  i=0, x=1 -> g = {1: [0]}
  i=1, x=8 -> g = {1: [0], 8: [1]}
  i=2, x=1 -> g = {1: [0, 2], 8: [1]}
  i=3, x=5 -> g = {1: [0, 2], 8: [1], 5: [3]}
  i=4, x=1 -> g = {1: [0, 2, 4], 8: [1], 5: [3]}
  i=5, x=5 -> g = {1: [0, 2, 4], 8: [1], 5: [3, 5]}
  i=6, x=8 -> g = {1: [0, 2, 4], 8: [1, 6], 5: [3, 5]}
  i=7, x=5 -> g = {1: [0, 2, 4], 8: [1, 6], 5: [3, 5, 7]}
  x=1: pos=[0, 2, 4], i1+i3=4, 2*i2=4 -> cách đều, ĐẶC BIỆT
  x=8: pos=[1, 6] có 2 lần != 3 -> không đặc biệt
  x=5: pos=[3, 5, 7], i1+i3=10, 2*i2=10 -> cách đều, ĐẶC BIỆT
  => đáp án: 2


In [15]:
def trace_counters(nums):
    print("nums =", nums)
    count, first, last, gap, ok = {}, {}, {}, {}, {}
    for i, x in enumerate(nums):
        count[x] = count.get(x, 0) + 1
        c = count[x]
        if c == 1:
            first[x] = last[x] = i
            note = f"lần 1 tại {i}"
        elif c == 2:
            gap[x] = i - first[x]
            last[x] = i
            note = f"lần 2 tại {i}, gap = {gap[x]}"
        elif c == 3:
            ok[x] = (i - last[x] == gap[x])
            last[x] = i
            note = f"lần 3 tại {i}: khoảng cách mới = {i - (first[x] + gap[x])}, gap = {gap[x]} -> ok = {ok[x]}"
        else:
            note = f"lần {c}: vượt 3, bị loại"
        print(f"  i={i}, x={x}: {note}")
    answer = sum(1 for x in count if count[x] == 3 and ok.get(x, False))
    print("  => đáp án:", answer)
    return answer


assert trace_counters([8, 6, 6, 8, 8]) == 0

nums = [8, 6, 6, 8, 8]
  i=0, x=8: lần 1 tại 0
  i=1, x=6: lần 1 tại 1
  i=2, x=6: lần 2 tại 2, gap = 1
  i=3, x=8: lần 2 tại 3, gap = 3
  i=4, x=8: lần 3 tại 4: khoảng cách mới = 1, gap = 3 -> ok = False
  => đáp án: 0


Bảng tóm tắt (Cách 2, `nums = [8,6,6,8,8]`):

| `i` | `x` | `g` sau khi thêm |
|---|---|---|
| 0 | 8 | `{8: [0]}` |
| 1 | 6 | `{8: [0], 6: [1]}` |
| 2 | 6 | `{8: [0], 6: [1, 2]}` |
| 3 | 8 | `{8: [0, 3], 6: [1, 2]}` |
| 4 | 8 | `{8: [0, 3, 4], 6: [1, 2]}` |

Sau đó: `6` có 2 lần (loại), `8` có đúng 3 lần nhưng `0 + 4 = 4 ≠ 2·3 = 6`, không đều. Đáp án `0`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (dict các danh sách chỉ số). Đúng class `Solution`, đúng method `countSpecialIntegers(self, nums: List[int]) -> int`.

In [16]:
from collections import defaultdict
from typing import List


class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        g = defaultdict(list)                     # giá trị -> các chỉ số (tăng dần)
        for i, x in enumerate(nums):
            g[x].append(i)
        answer = 0
        for pos in g.values():
            # đúng 3 lần, rồi mới đọc pos[2]; cách đều <=> i1 + i3 == 2 * i2
            if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1]:
                answer += 1
        return answer


for nums, expected in examples:
    assert Solution().countSpecialIntegers(nums) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Gom chỉ số theo giá trị:** duyệt `i` tăng dần nên mỗi `g[x]` là danh sách các chỉ số của `x` **sắp tăng**. Vậy khi `len(g[x]) == 3`, `pos[0] < pos[1] < pos[2]` chính là `i1 < i2 < i3` của đề.
- **"Đúng 3 lần":** điều kiện `len(pos) == 3` kiểm tra đúng yêu cầu này; các giá trị xuất hiện 1, 2 hoặc ≥ 4 lần bị loại.
- **"Cách đều":** `i2 − i1 = i3 − i2` tương đương `i1 + i3 = 2·i2` (cộng hai vế với `i2 + i1`). Phép kiểm tra không phụ thuộc giá trị `x`.
- **Đếm các giá trị khác nhau:** mỗi khóa của `g` là một giá trị khác nhau, và ta cộng tối đa 1 cho mỗi khóa, nên không đếm trùng.
- **Cách 3 tương đương:** gap thứ nhất là `pos[1] − pos[0]`; kiểm tra gap thứ hai bằng gap thứ nhất chính là `pos[2] − pos[1] == pos[1] − pos[0]`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d` là số giá trị khác nhau (`d ≤ n`).

- **Cách 1 (quét lại mỗi giá trị):** Time `O(n·d)` (tệ nhất `O(n²)`), Space `O(d)`.
- **Cách 2 (dict danh sách):** Time `O(n)` trung bình (hash `O(1)`), Space `O(n)`.
- **Cách 3 (đếm một lượt):** Time `O(n)`, Space `O(d)`.
- **Cách 4 (sắp xếp + groupby):** Time `O(n log n)`, Space `O(n)`.
- Với mọi cách phải đọc hết `nums` nên không thể nhanh hơn `O(n)`. Do `n ≤ 100` và giá trị ≤ 100, mọi cách đều rất nhanh.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` và viết một **lời giải độc lập** `oracle` duyệt **mọi bộ ba chỉ số** `(a < b < c)` cùng giá trị, kết hợp `nums.count(x) == 3`, hoàn toàn khác cách gom nhóm ở trên.

In [17]:
sol = Solution()
impls = {
    "bruteforce": count_special_bruteforce,
    "dict": count_special_dict,
    "counters": count_special_counters,
    "sort": count_special_sort,
    "submit": sol.countSpecialIntegers,
}


def oracle(nums):
    """Đáp án độc lập: duyệt mọi bộ ba chỉ số cùng giá trị cách đều, giá trị đó xuất hiện đúng 3 lần."""
    n = len(nums)
    special = set()
    for a in range(n):
        for b in range(a + 1, n):
            for c in range(b + 1, n):
                if nums[a] == nums[b] == nums[c] and b - a == c - b and nums.count(nums[a]) == 3:
                    special.add(nums[a])
    return len(special)


for nums, expected in examples:
    assert oracle(nums) == expected
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    assert oracle(nums) == expected, (label, "oracle", oracle(nums), expected)
    for name, fn in impls.items():
        got = fn(nums)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 1, 1], 1, "n = 3 nhỏ nhất, ba chỉ số liên tiếp 0,1,2")
check_all([1, 2, 3], 0, "mọi giá trị khác nhau")
check_all([5, 5, 5, 5], 0, "xuất hiện 4 lần")
check_all([5, 5, 5, 5, 5], 0, "xuất hiện 5 lần (còn cách đều nhưng không phải đúng 3)")
check_all([1, 1, 2, 1], 0, "chỉ số 0,1,3: không đều")
check_all([1, 2, 1, 2, 1, 2], 2, "cả 1 (0,2,4) và 2 (1,3,5) đều đặc biệt")
check_all([7, 1, 1, 7, 1, 1, 7], 1, "7 tại 0,3,6 cách đều 3; 1 xuất hiện 4 lần")
check_all([3, 4, 3, 4, 4, 3], 0, "3 tại 0,2,5 (không đều); 4 tại 1,3,4 (không đều)")
check_all([100] * 3 + [1] * 3, 2, "giá trị biên 100 và 1: 100 tại 0,1,2; 1 tại 3,4,5")
check_all([9] * 100, 0, "n = 100 toàn một giá trị")

PASS n = 3 nhỏ nhất, ba chỉ số liên tiếp 0,1,2 -> 1
PASS mọi giá trị khác nhau -> 0
PASS xuất hiện 4 lần -> 0
PASS xuất hiện 5 lần (còn cách đều nhưng không phải đúng 3) -> 0
PASS chỉ số 0,1,3: không đều -> 0
PASS cả 1 (0,2,4) và 2 (1,3,5) đều đặc biệt -> 2
PASS 7 tại 0,3,6 cách đều 3; 1 xuất hiện 4 lần -> 1
PASS 3 tại 0,2,5 (không đều); 4 tại 1,3,4 (không đều) -> 0
PASS giá trị biên 100 và 1: 100 tại 0,1,2; 1 tại 3,4,5 -> 2
PASS n = 100 toàn một giá trị -> 0


### 12.2. Tính chất và phản ví dụ

In [19]:
# Tính chất 1: đổi tên giá trị (song ánh) không đổi đáp án; đảo ngược mảng cũng không đổi đáp án
random.seed(4048)
for _ in range(300):
    n = random.randint(3, 14)
    nums = [random.randint(1, 4) for _ in range(n)]
    perm = random.sample(range(1, 101), 4)
    renamed = [perm[x - 1] for x in nums]
    assert sol.countSpecialIntegers(nums) == sol.countSpecialIntegers(renamed) == sol.countSpecialIntegers(nums[::-1])
print("PASS tính chất: đổi tên giá trị / đảo ngược mảng không đổi đáp án")

# Tính chất 2: đáp án <= n // 3 (mỗi giá trị đặc biệt chiếm đúng 3 vị trí)
for _ in range(300):
    n = random.randint(3, 30)
    nums = [random.randint(1, 5) for _ in range(n)]
    assert sol.countSpecialIntegers(nums) <= n // 3
print("PASS tính chất: đáp án <= n // 3")

# Phản ví dụ: 'ít nhất 3 lần' làm đáp án sai
def sai_it_nhat_3(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    return sum(len(p) >= 3 and p[0] + p[2] == 2 * p[1] for p in g.values())

print("[8,8,8,8] -> sai:", sai_it_nhat_3([8, 8, 8, 8]), "| đúng:", sol.countSpecialIntegers([8, 8, 8, 8]))
assert sai_it_nhat_3([8, 8, 8, 8]) == 1 and sol.countSpecialIntegers([8, 8, 8, 8]) == 0

PASS tính chất: đổi tên giá trị / đảo ngược mảng không đổi đáp án
PASS tính chất: đáp án <= n // 3
[8,8,8,8] -> sai: 1 | đúng: 0


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(3, 14)
    hi = random.choice([2, 3, 4, 6, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)

# mảng dài tới 100: oracle ba vòng lặp vẫn đủ nhanh (100^3 / 6), dùng ít trường hợp hơn
for trial in range(40):
    n = random.randint(60, 100)
    hi = random.choice([5, 10, 20])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums)
print("PASS 3040 input ngẫu nhiên: mọi cách khớp oracle độc lập")

PASS 3040 input ngẫu nhiên: mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng "ít nhất 3 lần" thay vì "đúng 3 lần"

In [21]:
def sai_it_nhat_ba(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    # chỉ kiểm tra ba lần đầu: bỏ qua việc x còn xuất hiện nữa
    return sum(len(p) >= 3 and p[0] + p[2] == 2 * p[1] for p in g.values())


print("sai:", sai_it_nhat_ba([8, 8, 8, 8]), "| đúng:", Solution().countSpecialIntegers([8, 8, 8, 8]))

sai: 1 | đúng: 0


**Vì sao sai:** đề yêu cầu **đúng ba lần**. Giá trị `8` xuất hiện bốn lần nên không đặc biệt, dù ba lần đầu (`0, 1, 2`) cách đều. Phải dùng `len(pos) == 3`.

### Lỗi 2: Đọc `pos[2]` khi danh sách ngắn hơn 3

In [22]:
def sai_index_error(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    count = 0
    for pos in g.values():
        if pos[0] + pos[2] == 2 * pos[1] and len(pos) == 3:   # kiểm tra độ dài SAU khi đã đọc pos[2]
            count += 1
    return count


try:
    sai_index_error([8, 6, 6, 8, 8])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** biểu thức `and` đánh giá **từ trái sang phải** và dừng khi gặp `False`. Nếu đặt `len(pos) == 3` ở cuối thì `pos[2]` đã bị đọc trước, gây `IndexError` khi `pos` chỉ có 1 hoặc 2 phần tử. Luôn kiểm tra độ dài **trước**.

### Lỗi 3: So sánh giá trị thay vì chỉ số

In [23]:
def sai_so_gia_tri(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(x)                 # nhầm: lưu giá trị x thay vì chỉ số i
    return sum(len(p) == 3 and p[0] + p[2] == 2 * p[1] for p in g.values())


print("sai:", sai_so_gia_tri([8, 6, 6, 8, 8]), "| đúng:", Solution().countSpecialIntegers([8, 6, 6, 8, 8]))

sai: 1 | đúng: 0


**Vì sao sai:** "cách đều" nói về **vị trí** (chỉ số), không phải giá trị. Nếu lưu `x` thì `8 + 8 == 2·8` luôn đúng, nên mọi giá trị xuất hiện 3 lần đều bị coi là đặc biệt. Ví dụ 3 cho `1` thay vì `0`.

### Lỗi 4: Đếm số lần thay vì đếm số giá trị khác nhau

In [24]:
def sai_dem_so_lan(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    return sum(len(p) for p in g.values() if len(p) == 3 and p[0] + p[2] == 2 * p[1])


print("sai:", sai_dem_so_lan([1, 8, 1, 5, 1, 5, 8, 5]), "| đúng:", Solution().countSpecialIntegers([1, 8, 1, 5, 1, 5, 8, 5]))

sai: 6 | đúng: 2


**Vì sao sai:** đề hỏi "bao nhiêu **số nguyên đặc biệt khác nhau**", mỗi số chỉ đếm 1. Cộng `len(p)` cho `3` mỗi số làm ví dụ 1 ra `6` thay vì `2`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Đề nói "số lần xuất hiện" hoặc "vị trí các lần xuất hiện của mỗi giá trị": **gom theo giá trị** bằng `dict` / `defaultdict(list)` / `Counter`.
- Cần cả **vị trí** và **số lần**: lưu `list` chỉ số, vì độ dài list chính là số lần.
- "Cách đều / cấp số cộng" của ba số `a < b < c`: kiểm tra `a + c == 2 * b`, tránh phép chia và số thực.
- Ràng buộc `nums[i] ≤ 100` và `n ≤ 100` cho phép gần như mọi cách, nhưng hãy chọn cách `O(n)` bằng dict để luyện thói quen.
- Dùng `and` ngắn mạch để **kiểm tra độ dài trước, đọc phần tử sau**.
- Luôn dựng một oracle ba vòng lặp lồng nhau trên mảng nhỏ để kiểm tra đáp án: nó khác hẳn cách gom nhóm nên bắt được lỗi logic.

## 15. Biến thể của bài

1. **Tổng quát số lần:** đếm số giá trị xuất hiện **đúng `m` lần** và các chỉ số tạo thành **cấp số cộng** (mọi khoảng cách liên tiếp bằng nhau).
2. **Trả về các giá trị đặc biệt** (đã sắp xếp) cùng khoảng cách chung, thay vì chỉ đếm.
3. **Đúng 3 lần, không yêu cầu cách đều** — chỉ là `Counter`.
4. **Hai lần cách nhau đúng `d`**: giá trị xuất hiện đúng 2 lần, khoảng cách bằng `d`.
5. **Cách đều theo giá trị** (ba giá trị liên tiếp tạo cấp số cộng) — khác hẳn bài này, thuộc nhóm hash set / sort.

In [25]:
def count_equally_spaced(nums: list[int], m: int) -> int:
    """Biến thể 1: số giá trị xuất hiện đúng m lần với các chỉ số tạo thành cấp số cộng."""
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    answer = 0
    for pos in g.values():
        if len(pos) == m and len({pos[j + 1] - pos[j] for j in range(m - 1)}) <= 1:
            answer += 1
    return answer


def special_values(nums: list[int]) -> list[tuple[int, int]]:
    """Biến thể 2: danh sách (giá trị, khoảng cách chung) đã sắp xếp theo giá trị."""
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    return sorted((x, pos[1] - pos[0]) for x, pos in g.items()
                  if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1])


for nums, expected in examples:
    assert count_equally_spaced(nums, 3) == expected
print(special_values([1, 8, 1, 5, 1, 5, 8, 5]))        # [(1, 2), (5, 2)]
assert special_values([1, 8, 1, 5, 1, 5, 8, 5]) == [(1, 2), (5, 2)]
assert count_equally_spaced([4, 1, 4, 1, 4, 1, 4], 4) == 1    # 4 tại 0,2,4,6 (cấp số cộng); 1 có 3 lần nhưng m = 4
assert count_equally_spaced([4, 1, 4, 1, 4, 1, 4], 3) == 1    # 1 tại 1,3,5
assert count_equally_spaced([1, 2, 1, 1], 3) == 0
print("PASS biến thể")

[(1, 2), (5, 2)]
PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lại mỗi giá trị | `set(nums)` rồi quét `n` lần | `O(n·d)` | `O(d)` | hiểu đề; đối chiếu |
| 2. `dict` danh sách chỉ số | `g[x].append(i)`, kiểm tra `len == 3` và `i1+i3 == 2·i2` | `O(n)` | `O(n)` | **cách nộp bài** |
| 3. Đếm một lượt | nhớ `first`, `gap`, `count` cho mỗi giá trị | `O(n)` | `O(d)` | tiết kiệm bộ nhớ, dữ liệu streaming |
| 4. Sắp xếp + `groupby` | sắp cặp `(x, i)`, tách nhóm | `O(n log n)` | `O(n)` | không muốn dùng hash |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(enumerate("ab"))`, `Counter([2, 2, 3])[2]`, `0 + 6 == 2 * 3`, `Solution().countSpecialIntegers([5, 5, 5])`, `Solution().countSpecialIntegers([7, 7, 7, 7])`.

**Bài 2 (tính tay).** Với `nums = [2, 9, 2, 9, 2, 9, 9]`: liệt kê chỉ số của mỗi giá trị rồi cho biết giá trị nào đặc biệt. Làm lại với `nums = [4, 4, 1, 4]`.

**Bài 3 (điền TODO).** Hoàn thành `special_todo(nums)` theo Cách 2.

In [26]:
def special_todo(nums):
    # TODO 1: tạo g = defaultdict(list)
    # TODO 2: duyệt i, x bằng enumerate và thêm i vào g[x]
    # TODO 3: duyệt pos trong g.values(); nếu len(pos) == 3 và pos[0] + pos[2] == 2 * pos[1] thì đếm
    # TODO 4: trả về số lượng đếm được
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `special_groupby2(nums)` theo Cách 4 (sắp xếp + `groupby`) mà không nhìn code mẫu. Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `special_list(nums)` trả về **danh sách các giá trị đặc biệt** đã sắp tăng. (b) Viết `count_special_k(nums, m)` đếm số giá trị xuất hiện **đúng `m` lần** (`m ≥ 2`) với mọi khoảng cách giữa các lần xuất hiện liên tiếp **bằng nhau** (cấp số cộng).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `enumerate` cho cặp `(chỉ số, giá trị)`; `Counter` đếm số lần; ba số `5,5,5` ở chỉ số `0,1,2`. Bài 2: viết chỉ số phía trên mỗi số rồi gom theo giá trị.
- **Mức 2 (vừa):** Bài 1: `[(0, 'a'), (1, 'b')]`, `2`, `True`, `1`, `0`. Bài 2: `[2,9,2,9,2,9,9]`: `2` tại `0,2,4` (đều) nên đặc biệt; `9` tại `1,3,5,6` (4 lần) không. Đáp án `1`. `[4,4,1,4]`: `4` tại `0,1,3` (`0 + 3 = 3`, `2·1 = 2`) không đều, đáp án `0`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `pairs = sorted((x, i) for i, x in enumerate(nums))`; `for _, grp in groupby(pairs, key=lambda p: p[0])`. Bài 5(a): `sorted(x for x, pos in g.items() if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1])`. Bài 5(b): `len(pos) == m` và `len({pos[j+1] - pos[j] for j in range(m - 1)}) == 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert list(enumerate("ab")) == [(0, "a"), (1, "b")]
assert Counter([2, 2, 3])[2] == 2
assert (0 + 6 == 2 * 3) is True
assert Solution().countSpecialIntegers([5, 5, 5]) == 1
assert Solution().countSpecialIntegers([7, 7, 7, 7]) == 0

# Bài 2
assert [i for i, v in enumerate([2, 9, 2, 9, 2, 9, 9]) if v == 2] == [0, 2, 4]
assert [i for i, v in enumerate([2, 9, 2, 9, 2, 9, 9]) if v == 9] == [1, 3, 5, 6]
assert Solution().countSpecialIntegers([2, 9, 2, 9, 2, 9, 9]) == 1
assert Solution().countSpecialIntegers([4, 4, 1, 4]) == 0

# Bài 3
def special_todo_ans(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    count = 0
    for pos in g.values():
        if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1]:
            count += 1
    return count

for nums, expected in examples:
    assert special_todo_ans(nums) == expected

# Bài 4
def special_groupby2(nums):
    pairs = sorted((x, i) for i, x in enumerate(nums))
    count = 0
    for _, grp in groupby(pairs, key=lambda p: p[0]):
        pos = [i for _, i in grp]
        if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1]:
            count += 1
    return count

for nums, expected in examples:
    assert special_groupby2(nums) == expected

# Bài 5(a)
def special_list(nums):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    return sorted(x for x, pos in g.items() if len(pos) == 3 and pos[0] + pos[2] == 2 * pos[1])

assert special_list([1, 8, 1, 5, 1, 5, 8, 5]) == [1, 5]
assert special_list([8, 6, 6, 8, 8]) == []

# Bài 5(b)
def count_special_k(nums, m):
    g = defaultdict(list)
    for i, x in enumerate(nums):
        g[x].append(i)
    return sum(1 for pos in g.values()
               if len(pos) == m and len({pos[j + 1] - pos[j] for j in range(m - 1)}) == 1)

assert count_special_k([1, 8, 1, 5, 1, 5, 8, 5], 3) == 2
assert count_special_k([1, 8, 1, 5, 1, 5, 8, 5], 2) == 1     # chỉ có 8 xuất hiện đúng 2 lần
assert count_special_k([4, 1, 4, 1, 4, 1, 4], 4) == 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (gom chỉ số theo giá trị, kiểm tra độ dài rồi kiểm tra cấp số cộng)

```python
from collections import defaultdict
from typing import List


class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        g = defaultdict(list)
        for i, x in enumerate(nums):
            g[x].append(i)
        return sum(len(p) == 3 and p[0] + p[2] == 2 * p[1] for p in g.values())
```

**Checklist**

- [ ] Hiểu "đặc biệt" = **đúng 3 lần** và **chỉ số** cách đều (không phải giá trị)?
- [ ] Gom **chỉ số** (`i`) chứ không gom giá trị (`x`) vào danh sách?
- [ ] Kiểm tra `len(pos) == 3` **trước** khi đọc `pos[2]`?
- [ ] Dùng `pos[0] + pos[2] == 2 * pos[1]` cho "cách đều"?
- [ ] Đếm số **giá trị khác nhau**, không phải số lần?
- [ ] Đã thử: `[x, x, x]`, giá trị xuất hiện 4 lần, hai lần, ba lần không đều?
- [ ] Skeleton đúng `class Solution` / `def countSpecialIntegers(self, nums: List[int]) -> int`, có `import` cần thiết?